# Silver cleaning — Medidas (transformer hard limits)

This notebook extracts each transformer's hard limits from the bronze metadata table and
saves them as `hive_metastore.silver.silver_medidas`, with one row per transformer and two
columns: `H_LIM_C` (current limit) and `H_LIM_V` (voltage limit).

These limits are what the overload label is built on: a transformer is overloaded when its
current or voltage exceeds the matching hard limit.

The steps are:

1. **Load and select** the tag and its limit value.
2. **Parse the limit into a number** — the raw values are text and use mixed decimal
   conventions, so this step normalises them carefully.
3. **Filter** to the same tag patterns used in the signal notebook, so the limits line up
   with the transformers that have readings.
4. **Pivot** the two per-transformer limits (suffix `U--` and `I--`) into `H_LIM_V` and
   `H_LIM_C`, matching the voltage/current convention used for the signals.
5. **Tidy and save**.

The tag filtering and pivot mirror the ARQLMED silver notebook on purpose, so the limits
and the signals use the same transformer identifiers.

## 1. Load and select

Load the common functions, read the bronze metadata table, and keep just the tag and its
limit value.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/02_silver_cleaning/00_common_functions

In [0]:
df = spark.read.table("hive_metastore.bronze.bronze_medidas")
display(df.limit(5))
df.printSchema()

In [0]:
df = df.select(["TAG","ALR_H_LIM"])

In [0]:
display(df)

## 2. Parse the limit into a number

The raw `ALR_H_LIM` values are text and can use either decimal convention (comma or dot),
sometimes with thousands separators. This step works out which character is the decimal
mark in each value and converts it to a plain number, rather than assuming one format.

First, strip whitespace and any characters that are not digits, sign, comma, or dot.

In [0]:
col_name = "ALR_H_LIM"

df1 = (
    df
    # trim + remove whitespace anywhere (spaces, tabs, etc.)
    .withColumn("_s", F.regexp_replace(F.trim(F.col(col_name)), r"\s+", ""))
    # keep only digits, sign, comma, dot (drops currency symbols, letters, etc.)
    .withColumn("_s", F.regexp_replace(F.col("_s"), r"[^0-9\+\-\,\.]", ""))
)

Find the last position of a dot and of a comma in each value. Whichever comes last is treated as the decimal mark.

In [0]:
# last position of '.' and ',' (0 if not present)
rev = F.reverse(F.col("_s"))

dot_from_end = F.instr(rev, ".")
comma_from_end = F.instr(rev, ",")

last_dot = F.when(dot_from_end > 0, F.length(F.col("_s")) - dot_from_end + 1).otherwise(F.lit(0))
last_comma = F.when(comma_from_end > 0, F.length(F.col("_s")) - comma_from_end + 1).otherwise(F.lit(0))

df2 = (
    df1
    .withColumn("_last_dot", last_dot)
    .withColumn("_last_comma", last_comma)
)

Normalise each value to a plain dot-decimal number and cast it. `try_cast` returns null
instead of failing on anything that still can't be read, so bad values surface rather than
crash the step.

In [0]:
# normalize to a dot-decimal string
normalized = (
    F.when((F.col("_s") == "") | F.col("_s").isNull(), F.lit(None))

    # both exist and dot is the last separator -> dot is decimal, commas are thousands
    .when((F.col("_last_dot") > 0) & (F.col("_last_comma") > 0) & (F.col("_last_dot") > F.col("_last_comma")),
          F.regexp_replace(F.col("_s"), ",", ""))

    # both exist and comma is the last separator -> comma is decimal, dots are thousands
    .when((F.col("_last_dot") > 0) & (F.col("_last_comma") > 0) & (F.col("_last_comma") > F.col("_last_dot")),
          F.regexp_replace(F.regexp_replace(F.col("_s"), r"\.", ""), ",", "."))

    # only comma exists -> comma is decimal
    .when((F.col("_last_comma") > 0) & (F.col("_last_dot") == 0),
          F.regexp_replace(F.col("_s"), ",", "."))

    # only dot exists (or none) -> already ok
    .otherwise(F.col("_s"))
)

df3 = (
    df2
    .withColumn("_norm", normalized)
    .withColumn(f"{col_name}_double", F.expr("try_cast(_norm as double)"))
    .drop("_s", "_last_dot", "_last_comma", "_norm")
)

display(df3)

Check which non-empty values failed to parse. This is a validation step — a clean run
should return few or no rows.

In [0]:
bad = (
    df3
    .filter(
        F.col(f"{col_name}_double").isNull() &
        F.col(col_name).isNotNull() &
        (F.trim(F.col(col_name)) != "")
    )
    .select(col_name)
    .distinct()
)

display(bad)

Replace the text limit with the parsed number.

In [0]:
df_med = (
    df3
    .withColumn("ALR_H_LIM", F.col("ALR_H_LIM_double"))
    .drop("ALR_H_LIM_double")
)

display(df_med)

In [0]:
df_med.count()

258,378 rows at this point.

## 3. Filter to matching transformers

Apply the same tag-pattern filter as the signal notebook (position 2 is `P` or `S`,
position 7 is not `-`, `9`, or `4`), then keep only tags ending in `U--` or `0II--`. This
keeps the limits aligned with the transformers that have signal readings.

In [0]:
df_med = df_med.filter(
    (col("TAG").substr(2, 1).isin("P", "S")) &  # Check position 2
    (~col("TAG").substr(7, 1).isin("-", "9", "4"))  # Check position 7
)

In [0]:
# Generate df_arqlmed_U- using filter and like operations
df_med = df_med.filter((col("TAG").rlike("U--$")) | (col("TAG").rlike("0II--$")))


In [0]:
df_med.count()

23,692 rows remain after filtering.

## 4. Pivot the limits into columns

Keep tag prefixes that report more than one limit signal, then pivot the two suffixes into
two columns: `U--` becomes `H_LIM_V` (voltage limit) and `I--` becomes `H_LIM_C` (current
limit). This matches the voltage/current convention used for the signals in the ARQLMED
silver notebook, so each limit lines up with the right signal downstream.

In [0]:
df_med = df_med.withColumn("TAG_prefix", substring(col("TAG"), 1, 11))

In [0]:
# Group by the first 12 characters and filter groups with more than one distinct ID
temp_df = df_med.groupBy("TAG_prefix").agg(count_distinct("TAG").alias("distinct_count")) \
    .filter(col("distinct_count") > 1)

# Join back with the original DataFrame to filter the relevant rows
df_med = df_med.join(temp_df, "TAG_prefix")

# Show the result
df_med.display()

In [0]:
p_df = df_med.withColumn("TAG_prefix", substring(col("TAG"), 1, 12)) \
           .withColumn("suffix", substring(col("TAG"), -3, 3))

# List of columns to pivot
columns_to_pivot = ["ALR_H_LIM"]


pivoted_dfs = []
for column in columns_to_pivot:
    pivoted_df = p_df.groupBy("TAG_prefix").pivot("suffix").agg(F.first(column))
    
    # Check the column names in the pivoted DataFrame
    print(pivoted_df.columns)
    
    # Rename columns based on expected pivot values
    pivoted_df = pivoted_df.withColumnRenamed("U--", f"{column}_T").withColumnRenamed("I--", f"{column}_I")
    
    # Ensure that renaming reflects actual column names after pivot
    if 'ALR_H_LIM_T' in pivoted_df.columns and 'ALR_H_LIM_I' in pivoted_df.columns:
        pivoted_df = pivoted_df.withColumnRenamed("ALR_H_LIM_T", "H_LIM_V").withColumnRenamed("ALR_H_LIM_I", "H_LIM_C")
    
    pivoted_dfs.append(pivoted_df)

In [0]:
display(pivoted_df)

## 5. Tidy and save

Drop rows missing either limit, round the limits, and save.

In [0]:
pivoted_df = pivoted_df.dropna(how='any')

Round both limits to 2 decimals.

In [0]:
pivoted_df = pivoted_df.withColumn("H_LIM_C", round(col("H_LIM_C"), 2)) \
                       .withColumn("H_LIM_V", round(col("H_LIM_V"), 2))

display(pivoted_df)

Profile the result.

In [0]:
dbutils.data.summarize(pivoted_df)

Save the limits table as Delta. `overwrite` plus `overwriteSchema` makes the cell safely
re-runnable.

**Target table:** `hive_metastore.silver.silver_medidas`

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "silver"
target_table = "silver_medidas"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

Save the table.

> Minor: this cell reads back with `spark.table(...)`, while cell 1 and the rest of the project use `spark.read.table(...)`.

In [0]:
(
    pivoted_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))


Read the saved table back and profile it as a final check.

In [0]:
df = spark.read.table("hive_metastore.silver.silver_medidas")
display(df.limit(5))
df.printSchema()

In [0]:
dbutils.data.summarize(df)